In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# pytorch modules and dependencies
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataloader, Dataset

# import resnet50 pretrained model
from torchvision.models import resnet50, ResNet50_Weights

# reads DICOM medical files
import pydicom

# displays graphs and images
import matplotlib.pyplot as plt

### check device if CUDA is available if not use CPU

In [ ]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

print(device)

# load, explore and analyze the data provided

### load and analyze the train.csv

In [ ]:
# load the train csv data and describe
train = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
print(train.shape)
display(train.head(5))

# get the labels
LABELS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
print("\n", len(LABELS), LABELS)

In [ ]:
# check for unlabelled studies in the train data, which means they rely on the report
has_label = train[LABELS].notna().all(axis=1)
print(f"Labelled data: {has_label.sum()}")
print(f"Unlabelled data: {(~has_label).sum()}")

In [ ]:
# look at the occurence for the labelled studies to check for imbalance and plot the frequency of
# labelled studies positive in findings
frequency = train.loc[has_label, LABELS].mean().sort_values(ascending=False)
print("Fraction of labelled studies positive for each finding:")
print(frequency.round(3))
frequency.plot.barh(title="label frequency")

In [ ]:
# explore the reports inorder to extract the labels for unlabelled studies
for i in range(5):
    print(f"Report {i}")
    print(str(train.loc[i, "Report"])[:600], "\n")
# also check for studies with missing reports
print("Missing reports: ", train["Report"].isna().sum())

### load and analyze the train_series.csv

In [ ]:
# load and describe the train_series data
train_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')
print(train_series.shape)
display(train_series.head(5))
print("\nStudies in train_series table: ", train_series["StudyInstanceUID"].nunique())
print("\nSeries per study:")
print(train_series.groupby("StudyInstanceUID").size().describe())

# anatomical plane
print("\nPlane counts:")
print(train_series["Anatomical_Plane"].value_counts())

In [ ]:
# compare fluid sensitive and fat suppression columns to check for similarity
print(pd.crosstab(train_series["Fluid_Sensitive"], train_series["Fat_Suppression"]))

# check how many studies contain at least on fluid_sensitive series in each plane
fs = train_series[train_series["Fluid_Sensitive"] == 1]
print("\nStudies with fluid_sensitive series by plane:")
print(fs.groupby("Anatomical_Plane")["StudyInstanceUID"].nunique())

### load, explore and analyze the DICOM slices in the train_series folder

In [ ]:
def train_series_dir(split_dir, study_uid, series_uid):
    return ROOT/split_dir/study_uid/series_uid